# Verify row-level scorecard reconciliation and denominator meanings


Every input/output hash and reconciled cohort count is checked. Legacy arithmetic is not independent biological validation.

In [1]:
from pathlib import Path
import json, hashlib, ast
import pandas as pd
from starplast import record_scorecards as R, scorecard as SC, organisms as O
out = Path('results/record_reconciliation_2026_10_08_v3')
m = json.loads((out / 'manifest.json').read_text())
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
for path, digest in m['input_sha256'].items():
    assert sha(path) == digest, path
for name, digest in m['outputs'].items():
    assert sha(out / name) == digest, name
rows = pd.read_csv(out / 'cohort_reconciliation.csv')
assert len(rows) == 4112 and rows.eligible.sum() == 1038372
assert rows.standard_accuracy_matches_all_hidden.all() and rows.legacy_rate_matches_call_precision.all()
for path in ('starplast/record_scorecards.py', 'scripts/reconcile_record_scorecards.py'):
    ast.parse(Path(path).read_text(), feature_version=(3, 10))
m['summary']

{'legacy_rows': 1038372, 'distinct_cohorts': 4112, 'unique_organism_gene_addresses': 13860, 'cohorts_with_abstentions': 1852, 'different_all_hidden_vs_among_calls': 910, 'standard_metric_mismatches': 0, 'legacy_precision_mismatches': 0, 'biological_admission': 'not established; truth/exclusion/nested-fit lineage unresolved', 'uncertainty': 'unavailable without supplied biological groups; repeats not independent samples', 'runtime_inference_changes': 'none'}

A synthetic four-entity example fixes the interpretation: two correct, one wrong and one abstention. Class A also has a false positive from class B; it cannot disappear when opening the class card.

In [2]:
scope = R.RecordScope(O.TOXOPLASMA, 'feature_knn', 'fixture', SC.T_LABEL, 'fixed', 3, 'fixture', 'test', 'fixture', 'synthetic_control', 'gene', 'fixture only')
fixture = pd.DataFrame({'entity': ['a','b','c','d'], 'truth': ['A','A','B','B'], 'prediction': ['A',None,'A','B']})
card = R.aggregate(fixture, scope)
classes = R.class_cards(fixture, scope)
a = next(c for c in classes if c['class'] == 'A')
assert card['extra']['accuracy_all_hidden'] == 0.5
assert card['extra']['accuracy_among_calls'] == 2/3
assert a['class_metrics']['precision'] == 0.5 and a['class_metrics']['recall'] == 0.5
{'counts': card['counts'], 'all_hidden_accuracy': card['extra']['accuracy_all_hidden'], 'accuracy_among_calls': card['extra']['accuracy_among_calls'], 'class_A': a['class_metrics'], 'uncertainty': card['extra']['uncertainty']}

{'counts': {'eligible': 4, 'answered': 3, 'abstained': 1, 'correct': 2, 'wrong': 1, 'unique_biological_entities': 4}, 'all_hidden_accuracy': 0.5, 'accuracy_among_calls': 0.6666666666666666, 'class_A': {'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'true_positive': 1, 'false_positive': 1, 'false_negative': 1, 'evaluation_prevalence': 0.5}, 'uncertainty': {'status': 'unavailable', 'reason': 'Biological groups unresolved', 'biological_groups': None}}